# ResNet50V2 — Identity Mappings in Deep Residual Networks (TensorFlow / Keras)
**Paper:** Identity Mappings in Deep Residual Networks (He et al., ECCV 2016)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Block type | Pre-activation Bottleneck (BN→ReLU before conv) |
| Stages | 4 (depths: 3-4-6-3) |
| Final feature dim | 2048 |
| Parameters | ~25.6M |
| Top-1 (ImageNet) | ~75.6% |
| Input size | 224×224 |
| vs ResNet50 | Pre-activation order; cleaner identity shortcut; post BN+ReLU |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — ResNet50V2 Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _pre_act_bottleneck(x, filters, stride=1, conv_shortcut=False, name=None):
    """
    ResNet50V2 pre-activation bottleneck block.
    BN + ReLU happen BEFORE each convolution (pre-activation / full pre-activation).

    conv_shortcut=True  : projection shortcut — Conv1x1(preact) to match channels
    conv_shortcut=False : identity shortcut (or MaxPool if stride>1)
    """
    preact = layers.BatchNormalization(name=f'{name}_preact_bn')(x)
    preact = layers.Activation('relu', name=f'{name}_preact_relu')(preact)

    if conv_shortcut:
        shortcut = layers.Conv2D(
            filters * 4, 1, strides=stride, use_bias=False,
            name=f'{name}_0_conv')(preact)
    elif stride > 1:
        shortcut = layers.MaxPooling2D(1, strides=stride)(x)
    else:
        shortcut = x

    # 1x1 compress
    x = layers.Conv2D(filters, 1, use_bias=False, name=f'{name}_1_conv')(preact)
    x = layers.BatchNormalization(name=f'{name}_1_bn')(x)
    x = layers.Activation('relu', name=f'{name}_1_relu')(x)

    # 3x3 spatial
    x = layers.ZeroPadding2D(1, name=f'{name}_2_pad')(x)
    x = layers.Conv2D(filters, 3, strides=stride, use_bias=False,
                      name=f'{name}_2_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_2_bn')(x)
    x = layers.Activation('relu', name=f'{name}_2_relu')(x)

    # 1x1 expand (no BN/ReLU after — handled by next block's preact)
    x = layers.Conv2D(filters * 4, 1, use_bias=False, name=f'{name}_3_conv')(x)

    x = layers.Add(name=f'{name}_out')([shortcut, x])
    return x


def build_resnet50v2(num_classes=1000, input_shape=(224, 224, 3)):
    """
    ResNet50V2 — Identity Mappings in Deep Residual Networks.
    Paper: He et al., ECCV 2016.

    Key differences from ResNet50:
      1. Pre-activation blocks: BN + ReLU BEFORE each conv (not after)
      2. Stem conv has NO BN or ReLU (first block's preact handles it)
      3. Final post-activation BN + ReLU before GlobalAvgPool
      4. Shortcut path is a true identity when channels match

    Architecture (pre-activation Bottleneck, expansion=4):
      Stem    : Conv7x7/2 (no BN/ReLU) + MaxPool3x3/2  ->  64 x 56x56
      Stage 1 : 3 x PreActBottleneck(64)  stride=1      -> 256 x 56x56
      Stage 2 : 4 x PreActBottleneck(128) stride=2      -> 512 x 28x28
      Stage 3 : 6 x PreActBottleneck(256) stride=2      ->1024 x 14x14
      Stage 4 : 3 x PreActBottleneck(512) stride=2      ->2048 x  7x7
      Post    : BN + ReLU
      Head    : GlobalAvgPool -> Dense(num_classes)
    """
    inputs = keras.Input(shape=input_shape)

    # ── Stem — no BN/ReLU after conv (first block preact handles it) ──
    x = layers.ZeroPadding2D(3, name='conv1_pad')(inputs)
    x = layers.Conv2D(64, 7, strides=2, use_bias=False, name='conv1_conv')(x)  # 112x112
    x = layers.ZeroPadding2D(1, name='pool1_pad')(x)
    x = layers.MaxPooling2D(3, strides=2, name='pool1_pool')(x)                # 56x56

    # ── Stage 1: 3 blocks, 64 filters ──
    x = _pre_act_bottleneck(x, 64, stride=1, conv_shortcut=True,  name='conv2_block1')  # 256 ch
    x = _pre_act_bottleneck(x, 64, stride=1, conv_shortcut=False, name='conv2_block2')
    x = _pre_act_bottleneck(x, 64, stride=1, conv_shortcut=False, name='conv2_block3')

    # ── Stage 2: 4 blocks, 128 filters, stride=2 ──
    x = _pre_act_bottleneck(x, 128, stride=2, conv_shortcut=True,  name='conv3_block1')  # 512 ch, 28x28
    x = _pre_act_bottleneck(x, 128, stride=1, conv_shortcut=False, name='conv3_block2')
    x = _pre_act_bottleneck(x, 128, stride=1, conv_shortcut=False, name='conv3_block3')
    x = _pre_act_bottleneck(x, 128, stride=1, conv_shortcut=False, name='conv3_block4')

    # ── Stage 3: 6 blocks, 256 filters, stride=2 ──
    x = _pre_act_bottleneck(x, 256, stride=2, conv_shortcut=True,  name='conv4_block1')  # 1024 ch, 14x14
    x = _pre_act_bottleneck(x, 256, stride=1, conv_shortcut=False, name='conv4_block2')
    x = _pre_act_bottleneck(x, 256, stride=1, conv_shortcut=False, name='conv4_block3')
    x = _pre_act_bottleneck(x, 256, stride=1, conv_shortcut=False, name='conv4_block4')
    x = _pre_act_bottleneck(x, 256, stride=1, conv_shortcut=False, name='conv4_block5')
    x = _pre_act_bottleneck(x, 256, stride=1, conv_shortcut=False, name='conv4_block6')

    # ── Stage 4: 3 blocks, 512 filters, stride=2 ──
    x = _pre_act_bottleneck(x, 512, stride=2, conv_shortcut=True,  name='conv5_block1')  # 2048 ch, 7x7
    x = _pre_act_bottleneck(x, 512, stride=1, conv_shortcut=False, name='conv5_block2')
    x = _pre_act_bottleneck(x, 512, stride=1, conv_shortcut=False, name='conv5_block3')

    # ── Post-activation (last conv block has no trailing BN/ReLU) ──
    x = layers.BatchNormalization(name='post_bn')(x)
    x = layers.Activation('relu', name='post_relu')(x)

    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='resnet50v2')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_resnet50v2(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=120)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val',
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'resnet50v2_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)
print('Best model saved: resnet50v2_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ResNet50V2 — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
from PIL import Image

def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    labels    = [CLASS_NAMES[i] for i in top_idx]
    top_probs = [probs[i] * 100 for i in top_idx]
    axes[1].barh(labels[::-1], top_probs[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')

macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ResNet50V2 — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')